# 02 — VADER sentiment and divergence scoring

Computes per-review VADER compound score and the sentiment-star divergence used as a proxy label by the XGBoost classifier downstream. Output saved to `outputs/vader_scores.csv`.

VADER is a lexicon and rule-based sentiment tool, fast and deterministic, no model weights required. The divergence formula is:

$$\mathrm{divergence} = | \mathrm{vader\_compound} - \frac{\mathrm{stars} - 3}{2} |$$

Where the right-hand term normalizes stars from 1..5 to -1..+1 to align with VADER's range.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from src.data_loader import load_reviews
from src.config import OUTPUTS_DIR, DIVERGENCE_THRESHOLD

In [2]:
df = load_reviews(clean=True)
vader = SentimentIntensityAnalyzer()
df['vader_compound'] = df['text'].fillna('').map(
    lambda t: vader.polarity_scores(t)['compound']
)
df['normalized_stars'] = (df['stars'] - 3) / 2.0
df['sentiment_star_divergence'] = (df['vader_compound'] - df['normalized_stars']).abs()
df['suspicious'] = (df['sentiment_star_divergence'] > DIVERGENCE_THRESHOLD).astype(int)

{"receipt_type": "ingest", "ts": "2026-05-02T18:00:17.050153+00:00", "tenant_id": "cis509-mcook20", "source_type": "csv", "source_path": "data/restaurant_reviews_az.csv", "row_count": 47035, "redactions": []}


## Class balance and threshold sanity check

In [3]:
n_suspicious = int(df['suspicious'].sum())
pct = 100.0 * n_suspicious / len(df)
print(f'Threshold: |divergence| > {DIVERGENCE_THRESHOLD}')
print(f'Suspicious reviews: {n_suspicious:,} of {len(df):,} ({pct:.1f}%)')
print()
print('Mean divergence by star:')
print(df.groupby('stars')['sentiment_star_divergence'].mean().round(3))
print()
print('Suspicious rate by star:')
print(df.groupby('stars')['suspicious'].mean().round(3))

Threshold: |divergence| > 1.0
Suspicious reviews: 4,838 of 47,035 (10.3%)

Mean divergence by star:
stars
1    0.789
2    0.804
3    0.704
4    0.410
5    0.110
Name: sentiment_star_divergence, dtype: float64

Suspicious rate by star:
stars
1    0.347
2    0.435
3    0.000
4    0.012
5    0.010
Name: suspicious, dtype: float64


## Save scores to disk for downstream notebooks

In [4]:
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
out = df[['review_id', 'stars', 'vader_compound', 'normalized_stars',
          'sentiment_star_divergence', 'suspicious']].copy()
out_path = OUTPUTS_DIR / 'vader_scores.csv'
out.to_csv(out_path, index=False)
print(f'Saved {len(out):,} rows to {out_path.relative_to(out_path.parent.parent)}')

Saved 47,035 rows to outputs/vader_scores.csv


## Notes

VADER is intentionally simple. It is the lexicon baseline. Misses include:

- Sarcasm, food-review tone (specifics like 'too sweet' that read positive lexically)
- Negation handled but with limits, e.g. 'not bad at all' can flip
- No domain adaptation, restaurant-specific vocabulary like 'mid' or 'fire' has no signal

These misses are what motivate adding BERTopic in notebook 03 and the LLM agreement check in notebook 04. The XGBoost classifier in notebook 05 then learns what combinations of features actually predict the divergence label.